# Distill Agent Tools (Amazon Strands)

This notebook builds the full proposed **Distill Agent** tool set for the Hyperdelivery
factory using the Amazon **Strands Agents SDK**, and wires them into a real
`strands.Agent` backed by `BedrockModel`, matching the pattern used in
`src/providers/agents/strands_factory.py` and `src/agent_runtime/distill/main.py`.

## Tools built (10 total)

**Ingestion (7)**
1. `pptx_extract_text`
2. `docx_extract_text`
3. `xlsx_extract_data`
4. `pdf_extract_text`
5. `image_extract_text` (OCR)
6. `jira_ticket_ingest`
7. `github_readme_ingest`

**Authoring / quality (3)**
8. `generate_question_set`
9. `create_fsd_from_template`
10. `score_completeness`

## What this notebook does
1. Installs/imports required libraries.
2. Generates small sample files (pptx/docx/xlsx/pdf/image) so every ingestion tool has
   something real to run against, with no external files needed.
3. Defines each tool as a `@tool`-decorated function.
4. Runs every tool once, standalone, to prove it works.
5. Assembles all 10 tools into a real `strands.Agent` with `BedrockModel`.
6. Runs one end-to-end invocation against Amazon Bedrock and prints the structured FSD
   output.

> Requires AWS credentials for the live Bedrock call in the last section (this repo's
> `AWS_PROFILE=ua-hyperdelivery` already works locally). Jira/GitHub ingestion tools are
> optional-credential the same way `src/tools/atlassian.py` gates Confluence/Jira - they
> degrade gracefully instead of failing if not configured.


## 1. Install and Import Required Libraries

Amazon Strands Agents SDK, Bedrock model client, and format-specific parsing libraries
(`python-pptx`, `python-docx`, `openpyxl`, `pypdf`, `pytesseract` + `Pillow` for OCR),
plus `requests` for Jira/GitHub ingestion.

In [9]:
import io
import json
import os
import re
import textwrap
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any

import openpyxl
import pytesseract
import requests
from docx import Document as DocxDocument
from PIL import Image, ImageDraw
from pptx import Presentation
from pptx.util import Inches
from pydantic import BaseModel
from pypdf import PdfReader
from reportlab.pdfgen import canvas
from strands import Agent, tool
from strands.models import BedrockModel

# Load the repo's .env (AWS_REGION, ATLASSIAN_*, etc.) so this notebook reuses the same
# local configuration as `make dev-distill`, without duplicating settings by hand.
try:
    from dotenv import load_dotenv

    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    load_dotenv(REPO_ROOT / ".env")
except ImportError:
    REPO_ROOT = Path.cwd()

WORKDIR = REPO_ROOT / "notebooks" / "sample_data"
WORKDIR.mkdir(parents=True, exist_ok=True)
print("Sample files folder:", WORKDIR)
print("Repo root detected as:", REPO_ROOT)


Sample files folder: /Users/aishwarya.raj/Documents/Projects/hyperdelivery/ua-hyperdelivery/notebooks/sample_data
Repo root detected as: /Users/aishwarya.raj/Documents/Projects/hyperdelivery/ua-hyperdelivery


## 2. Generate Sample Source Files

So every ingestion tool below can be tested against a real file with no external
dependencies, we generate one small sample file per format here: a PPTX, a DOCX, an
XLSX, a PDF, and a PNG image with rendered text (for OCR).

In [10]:
# --- PPTX sample -------------------------------------------------------------
pptx_path = WORKDIR / "sample_requirements.pptx"
prs = Presentation()
slide = prs.slides.add_slide(prs.slide_layouts[1])
slide.shapes.title.text = "Password Reset Requirements"
body = slide.placeholders[1].text_frame
body.text = "Users must be able to reset their password via email."
p = body.add_paragraph()
p.text = "Reset links must expire after 30 minutes and be single-use."
prs.save(pptx_path)

# --- DOCX sample -------------------------------------------------------------
docx_path = WORKDIR / "sample_requirements.docx"
doc = DocxDocument()
doc.add_heading("Functional Requirements", level=1)
doc.add_paragraph("The system shall allow a user to request a password reset email.")
doc.add_paragraph("The system shall log every password reset attempt for audit purposes.")
doc.save(docx_path)

# --- XLSX sample -------------------------------------------------------------
xlsx_path = WORKDIR / "sample_requirements.xlsx"
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Requirements"
ws.append(["Requirement ID", "Description", "Priority"])
ws.append(["REQ-1", "User can request a password reset via email", "High"])
ws.append(["REQ-2", "Reset link expires after 30 minutes", "High"])
wb.save(xlsx_path)

# --- PDF sample ---------------------------------------------------------------
pdf_path = WORKDIR / "sample_requirements.pdf"
c = canvas.Canvas(str(pdf_path))
text_obj = c.beginText(50, 750)
for line in [
    "Data Requirements",
    "The system must store the user's registered email address.",
    "The system must store a hashed, single-use reset token with an expiry timestamp.",
]:
    text_obj.textLine(line)
c.drawText(text_obj)
c.save()

# --- Image sample (for OCR) ----------------------------------------------------
image_path = WORKDIR / "sample_whiteboard.png"
img = Image.new("RGB", (700, 160), color="white")
draw = ImageDraw.Draw(img)
draw.text((10, 10), "NFR: Password reset emails must be delivered within 60 seconds.", fill="black")
draw.text((10, 60), "Edge case: unknown email should not reveal account existence.", fill="black")
img.save(image_path)

print("Generated sample files:")
for p in [pptx_path, docx_path, xlsx_path, pdf_path, image_path]:
    print(" -", p)


Generated sample files:
 - /Users/aishwarya.raj/Documents/Projects/hyperdelivery/ua-hyperdelivery/notebooks/sample_data/sample_requirements.pptx
 - /Users/aishwarya.raj/Documents/Projects/hyperdelivery/ua-hyperdelivery/notebooks/sample_data/sample_requirements.docx
 - /Users/aishwarya.raj/Documents/Projects/hyperdelivery/ua-hyperdelivery/notebooks/sample_data/sample_requirements.xlsx
 - /Users/aishwarya.raj/Documents/Projects/hyperdelivery/ua-hyperdelivery/notebooks/sample_data/sample_requirements.pdf
 - /Users/aishwarya.raj/Documents/Projects/hyperdelivery/ua-hyperdelivery/notebooks/sample_data/sample_whiteboard.png


## 3. Ingestion Tool 1/7 - `pptx_extract_text`

Extracts slide titles, body text, and speaker notes from a PowerPoint file.

In [11]:
@tool
def pptx_extract_text(path: str) -> str:
    """Extract slide titles, body text, and speaker notes from a PowerPoint (.pptx) file.
    Use this when a stakeholder shares requirements as a slide deck."""
    prs = Presentation(path)
    chunks: list[str] = []
    for i, slide in enumerate(prs.slides, start=1):
        slide_lines = [f"[Slide {i}]"]
        for shape in slide.shapes:
            if shape.has_text_frame and shape.text_frame.text.strip():
                slide_lines.append(shape.text_frame.text.strip())
        if slide.has_notes_slide:
            notes = slide.notes_slide.notes_text_frame.text.strip()
            if notes:
                slide_lines.append(f"[Notes] {notes}")
        chunks.append("\n".join(slide_lines))
    return "\n\n".join(chunks)


# Standalone test
print(pptx_extract_text(str(pptx_path)))


[Slide 1]
Password Reset Requirements
Users must be able to reset their password via email.
Reset links must expire after 30 minutes and be single-use.


## 4. Ingestion Tool 2/7 - `docx_extract_text`

Extracts paragraph and table text from a Word document.

In [12]:
@tool
def docx_extract_text(path: str) -> str:
    """Extract paragraph and table text from a Word (.docx) document.
    Use this when a stakeholder shares requirements as a Word document."""
    doc = DocxDocument(path)
    lines = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    for table in doc.tables:
        for row in table.rows:
            cells = [c.text.strip() for c in row.cells if c.text.strip()]
            if cells:
                lines.append(" | ".join(cells))
    return "\n".join(lines)


# Standalone test
print(docx_extract_text(str(docx_path)))


Functional Requirements
The system shall allow a user to request a password reset email.
The system shall log every password reset attempt for audit purposes.


## 5. Ingestion Tool 3/7 - `xlsx_extract_data`

Extracts tabular rows from every sheet in an Excel workbook, rendered as pipe-delimited
text so the LLM can read it as structured data.

In [13]:
@tool
def xlsx_extract_data(path: str) -> str:
    """Extract tabular rows from every sheet in an Excel (.xlsx) workbook.
    Use this when a stakeholder shares requirements as a spreadsheet."""
    wb = openpyxl.load_workbook(path, data_only=True)
    sections: list[str] = []
    for sheet in wb.worksheets:
        rows = []
        for row in sheet.iter_rows(values_only=True):
            values = [str(v) for v in row if v is not None]
            if values:
                rows.append(" | ".join(values))
        if rows:
            sections.append(f"[Sheet: {sheet.title}]\n" + "\n".join(rows))
    return "\n\n".join(sections)


# Standalone test
print(xlsx_extract_data(str(xlsx_path)))


[Sheet: Requirements]
Requirement ID | Description | Priority
REQ-1 | User can request a password reset via email | High
REQ-2 | Reset link expires after 30 minutes | High


## 6. Ingestion Tool 4/7 - `pdf_extract_text`

Extracts text per page from a PDF using `pypdf` (the same library already used in
`src/providers/storage/s3.py`).

In [14]:
@tool
def pdf_extract_text(path: str) -> str:
    """Extract text per page from a PDF file.
    Use this when a stakeholder shares requirements as a PDF document."""
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if text:
            pages.append(f"[Page {i}]\n{text}")
    return "\n\n".join(pages)


# Standalone test
print(pdf_extract_text(str(pdf_path)))


[Page 1]
Data Requirements
The system must store the user's registered email address.
The system must store a hashed, single-use reset token with an expiry timestamp.


## 7. Ingestion Tool 5/7 - `image_extract_text` (OCR)

Runs OCR (Tesseract via `pytesseract`) over an image - useful for whiteboard photos or
screenshots shared by stakeholders.

In [15]:
@tool
def image_extract_text(path: str) -> str:
    """Run OCR over an image (e.g., a whiteboard photo or screenshot) and return the
    extracted text. Use this when a stakeholder shares requirements as an image."""
    image = Image.open(path)
    return pytesseract.image_to_string(image).strip()


# Standalone test
print(image_extract_text(str(image_path)))


[NER Password resetemails must be delivered within 60 seconds,

Edge case: unknown emailshould not revealaccountexistence,


## 8. Ingestion Tool 6/7 - `jira_ticket_ingest`

Reads a Jira ticket's summary, description, and comments via the Jira REST API.
Follows the repo's existing convention (`src/tools/atlassian.py`): credentials come
from environment/settings, never from the model, and the tool degrades gracefully
(raises a clear error) instead of crashing the notebook if `ATLASSIAN_*` isn't
configured.

In [16]:
def _atlassian_settings() -> tuple[str, str, str] | None:
    base_url = os.environ.get("ATLASSIAN_BASE_URL")
    email = os.environ.get("ATLASSIAN_EMAIL")
    token = os.environ.get("ATLASSIAN_API_TOKEN")
    if not (base_url and email and token):
        return None
    return base_url.rstrip("/"), email, token


def _atlassian_verify_tls() -> str | bool:
    """Return Requests verify mode for Atlassian calls.

    Priority:
    1) ATLASSIAN_SSL_VERIFY=false|0|no disables cert verification (debug only)
    2) ATLASSIAN_CA_BUNDLE / REQUESTS_CA_BUNDLE / SSL_CERT_FILE custom CA bundle
    3) True (default Requests verification)
    """
    verify_flag = os.environ.get("ATLASSIAN_SSL_VERIFY", "true").strip().lower()
    if verify_flag in {"0", "false", "no"}:
        return False

    ca_bundle = (
        os.environ.get("ATLASSIAN_CA_BUNDLE")
        or os.environ.get("REQUESTS_CA_BUNDLE")
        or os.environ.get("SSL_CERT_FILE")
    )
    if ca_bundle:
        return ca_bundle

    return True


@tool
def jira_ticket_ingest(issue_key: str) -> str:
    """Fetch a Jira ticket's summary, description, and comments by issue key
    (e.g., 'ADVA-3011'). Requires ATLASSIAN_BASE_URL/EMAIL/API_TOKEN to be configured -
    the model only ever supplies an issue key, never credentials."""
    creds = _atlassian_settings()
    if creds is None:
        raise ValueError(
            "ATLASSIAN_BASE_URL/EMAIL/API_TOKEN must be configured to use jira_ticket_ingest"
        )
    base_url, email, token = creds
    verify = _atlassian_verify_tls()

    resp = requests.get(
        f"{base_url}/rest/api/3/issue/{issue_key}",
        auth=(email, token),
        params={"fields": "summary,description,comment"},
        timeout=15,
        verify=verify,
    )
    resp.raise_for_status()
    data = resp.json()["fields"]

    def _plain_text(adf: Any) -> str:
        """Flatten Atlassian Document Format content into plain text."""
        if not isinstance(adf, dict):
            return str(adf or "")
        texts: list[str] = []
        for node in adf.get("content", []):
            if node.get("type") == "text":
                texts.append(node.get("text", ""))
            elif "content" in node:
                texts.append(_plain_text(node))
        return " ".join(t for t in texts if t)

    lines = [f"Summary: {data.get('summary', '')}", f"Description: {_plain_text(data.get('description'))}"]
    for comment in data.get("comment", {}).get("comments", []):
        lines.append(f"Comment ({comment.get('author', {}).get('displayName', 'unknown')}): "
                      f"{_plain_text(comment.get('body'))}")
    return "\n".join(lines)


# Standalone test (gated - only calls out if credentials are configured)
if _atlassian_settings() is None:
    print("jira_ticket_ingest: skipped - ATLASSIAN_BASE_URL/EMAIL/API_TOKEN not configured in .env")
else:
    verify_mode = _atlassian_verify_tls()
    print(f"jira_ticket_ingest is configured. TLS verify mode: {verify_mode}")
    print("Call jira_ticket_ingest('YOUR-ISSUE-KEY') to test live.")

jira_ticket_ingest is configured. TLS verify mode: False
Call jira_ticket_ingest('YOUR-ISSUE-KEY') to test live.


## 9. Create a Live Jira Sample Ticket and Test Ingestion

This section creates a real Jira issue assigned to your current Jira user, then calls `jira_ticket_ingest` on the created key.

Set this in your `.env` first:
- `ATLASSIAN_PROJECT_KEY` (example: `ADVA`)

The code is safe-gated:
- If Atlassian credentials are missing, it prints guidance.
- If `ATLASSIAN_PROJECT_KEY` is missing, it prints guidance.
- It only creates a ticket when both are present.

In [12]:
def jira_get_myself() -> dict[str, Any]:
    """Return current Jira user profile to derive assignee account id."""
    creds = _atlassian_settings()
    if creds is None:
        raise ValueError("ATLASSIAN_BASE_URL/EMAIL/API_TOKEN must be configured")

    base_url, email, token = creds
    resp = requests.get(
        f"{base_url}/rest/api/3/myself",
        auth=(email, token),
        timeout=15,
        verify=_atlassian_verify_tls(),
    )
    resp.raise_for_status()
    return resp.json()


def create_jira_sample_ticket(project_key: str, assignee_account_id: str | None = None) -> str:
    """Create a sample Jira Task and return the created issue key."""
    creds = _atlassian_settings()
    if creds is None:
        raise ValueError("ATLASSIAN_BASE_URL/EMAIL/API_TOKEN must be configured")

    base_url, email, token = creds
    timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

    description_adf = {
        "type": "doc",
        "version": 1,
        "content": [
            {
                "type": "paragraph",
                "content": [
                    {
                        "type": "text",
                        "text": "Sample ticket created from distill_agent_tools.ipynb for live ingestion test.",
                    }
                ],
            },
            {
                "type": "paragraph",
                "content": [
                    {
                        "type": "text",
                        "text": f"Created at: {timestamp}",
                    }
                ],
            },
        ],
    }

    fields: dict[str, Any] = {
        "project": {"key": project_key},
        "summary": f"[Distill Notebook Test] Password reset requirements sample - {timestamp}",
        "issuetype": {"name": "Task"},
        "description": description_adf,
        "labels": ["distill-test", "automation-sample"],
    }

    if assignee_account_id:
        fields["assignee"] = {"id": assignee_account_id}

    resp = requests.post(
        f"{base_url}/rest/api/3/issue",
        auth=(email, token),
        json={"fields": fields},
        timeout=20,
        verify=_atlassian_verify_tls(),
    )
    resp.raise_for_status()
    return resp.json()["key"]


# Live test: create issue assigned to current user, then ingest it via the notebook tool.
creds = _atlassian_settings()
project_key = os.environ.get("ATLASSIAN_PROJECT_KEY")

if creds is None:
    print("Set ATLASSIAN_BASE_URL, ATLASSIAN_EMAIL, ATLASSIAN_API_TOKEN in .env, then rerun this cell.")
elif not project_key:
    print("Set ATLASSIAN_PROJECT_KEY in .env (for example: ADVA), then rerun this cell.")
else:
    try:
        myself = jira_get_myself()
        assignee_id = myself.get("accountId")
        display_name = myself.get("displayName", "unknown")

        created_key = create_jira_sample_ticket(project_key=project_key, assignee_account_id=assignee_id)
        print(f"Created Jira issue: {created_key} (assigned to: {display_name})")
        print("\nIngested content preview:\n")
        print(jira_ticket_ingest(created_key))
    except requests.exceptions.SSLError as exc:
        print("TLS verification failed while calling Jira.")
        print("Set one of these in .env and rerun Cell 3 first:")
        print(" - ATLASSIAN_CA_BUNDLE=/absolute/path/to/company-root-ca.pem")
        print(" - or REQUESTS_CA_BUNDLE=/absolute/path/to/company-root-ca.pem")
        print("Temporary debug only: ATLASSIAN_SSL_VERIFY=false")
        print(f"\nOriginal error: {exc}")

/Users/aishwarya.raj/Documents/Projects/hyperdelivery/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'underarmour.atlassian.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/aishwarya.raj/Documents/Projects/hyperdelivery/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'underarmour.atlassian.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Created Jira issue: ADVA-3021 (assigned to: Aishwarya Raj)

Ingested content preview:



/Users/aishwarya.raj/Documents/Projects/hyperdelivery/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'underarmour.atlassian.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Summary: [Distill Notebook Test] Password reset requirements sample - 2026-08-28 09:59:58 UTC
Description: Sample ticket created from distill_agent_tools.ipynb for live ingestion test. Created at: 2026-08-28 09:59:58 UTC


## 9. Ingestion Tool 7/7 - `github_readme_ingest`

Fetches a repository's README via the public GitHub API. Works without a token for
public repos (rate-limited); set `GITHUB_TOKEN` in the environment to raise the limit
and access private repos.

In [2]:
def _github_verify_tls() -> str | bool:
    """Return Requests verify mode for GitHub calls.

    Priority:
    1) GITHUB_SSL_VERIFY=false|0|no disables cert verification (debug only)
    2) GITHUB_CA_BUNDLE / REQUESTS_CA_BUNDLE / SSL_CERT_FILE custom CA bundle
    3) True (default Requests verification)
    """
    verify_flag = os.environ.get("GITHUB_SSL_VERIFY", "true").strip().lower()
    if verify_flag in {"0", "false", "no"}:
        return False

    ca_bundle = (
        os.environ.get("GITHUB_CA_BUNDLE")
        or os.environ.get("REQUESTS_CA_BUNDLE")
        or os.environ.get("SSL_CERT_FILE")
    )
    if ca_bundle:
        return ca_bundle

    return True


def _github_headers() -> dict[str, str]:
    headers = {"Accept": "application/vnd.github.raw+json"}
    token = os.environ.get("GITHUB_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"
    return headers


@tool
def github_readme_ingest(owner_repo: str) -> str:
    """Fetch a GitHub repository's README as plain text, given 'owner/repo'.

    Raises a clear error when the repository is private/inaccessible, token lacks
    permissions, or README is missing from the root.
    """
    verify = _github_verify_tls()
    headers = _github_headers()

    # First check repository visibility/access so 404 can be explained clearly.
    repo_resp = requests.get(
        f"https://api.github.com/repos/{owner_repo}",
        headers=headers,
        timeout=15,
        verify=verify,
    )

    if repo_resp.status_code == 404:
        raise ValueError(
            f"Repository '{owner_repo}' is not accessible. For private repos, set GITHUB_TOKEN "
            "with repo access and authorize SSO for your org if required."
        )
    repo_resp.raise_for_status()

    readme_resp = requests.get(
        f"https://api.github.com/repos/{owner_repo}/readme",
        headers=headers,
        timeout=15,
        verify=verify,
    )

    if readme_resp.status_code == 404:
        raise ValueError(
            f"README not found for '{owner_repo}' at repository root via GitHub API /readme endpoint."
        )

    readme_resp.raise_for_status()
    return readme_resp.text


# Standalone test against a small, well-known public repo
try:
    readme_text = github_readme_ingest("octocat/Hello-World")
    print(readme_text[:500])
except requests.exceptions.SSLError as exc:
    print("TLS verification failed while calling GitHub.")
    print("Set one of these in .env and rerun Cell 3 first:")
    print(" - GITHUB_CA_BUNDLE=/absolute/path/to/company-root-ca.pem")
    print(" - or REQUESTS_CA_BUNDLE=/absolute/path/to/company-root-ca.pem")
    print("Temporary debug only: GITHUB_SSL_VERIFY=false")
    print(f"\nOriginal error: {exc}")
except ValueError as exc:
    print(f"GitHub integration check failed: {exc}")
except requests.RequestException as exc:
    print(f"github_readme_ingest: skipped live call - {exc}")

/Users/aishwarya.raj/Documents/Projects/hyperdelivery/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.github.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/aishwarya.raj/Documents/Projects/hyperdelivery/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.github.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Hello World!



In [3]:
import os
print("token_set:", bool(os.environ.get("GITHUB_TOKEN")))
print("ssl_verify:", os.environ.get("GITHUB_SSL_VERIFY"))
print("ca_bundle:", os.environ.get("GITHUB_CA_BUNDLE") or os.environ.get("REQUESTS_CA_BUNDLE"))

token_set: True
ssl_verify: false
ca_bundle: None


In [4]:
# One-off live test for the requested repository.
# Keep TLS verification ON where possible; set GITHUB_CA_BUNDLE/REQUESTS_CA_BUNDLE in .env.
# Temporary debug only: uncomment the next line if corporate cert setup is still pending.
# os.environ["GITHUB_SSL_VERIFY"] = "false"

try:
    repo_readme = github_readme_ingest("underarmour/ua-ai-platform")
    print(repo_readme[:2000])
except Exception as exc:
    print(f"GitHub one-off test failed: {exc}")

/Users/aishwarya.raj/Documents/Projects/hyperdelivery/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.github.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


GitHub one-off test failed: Repository 'underarmour/ua-ai-platform' is not accessible. For private repos, set GITHUB_TOKEN with repo access and authorize SSO for your org if required.


## 10. Authoring Tool 1/3 - `generate_question_set`

Groups open questions by theme and assigns each to a named SME, using the assignment
strategy discussed for this project:
1. explicit `source_owner` (e.g., a Jira ticket's owner) wins first,
2. otherwise fall back to a static `theme -> SME` map,
3. otherwise route to a fallback triage queue with `low` confidence, flagged
   `correction_required` rather than silently dropped.

In [5]:
# Static theme -> SME map (Phase 1 recommendation - swap for a real directory lookup later)
THEME_TO_SME = {
    "business_process": "priya.n",
    "business_rules": "rohan.k",
    "data_requirements": "priya.n",
    "data_source_connector": "asha.d",
    "security_compliance": "security-triage",
    "nfr_performance": "platform-triage",
    "ux_workflow": "maya.s",
}
FALLBACK_SME = "gov-triage"


@tool
def generate_question_set(
    gaps: list[dict[str, str]],
) -> list[dict[str, Any]]:
    """Group open questions by theme and assign each to a named SME. Each item in
    `gaps` is a dict with 'theme', 'question', and optionally 'source_owner' (e.g., a
    Jira ticket assignee) taken from the ingested source. Call this whenever normalized
    input has ambiguity or missing detail - do not guess the answer yourself."""
    question_set = []
    for i, gap in enumerate(gaps, start=1):
        theme = gap.get("theme", "unknown")
        source_owner = gap.get("source_owner")
        if source_owner:
            assignee, reason, confidence = source_owner, "source_owner", "high"
        elif theme in THEME_TO_SME:
            assignee, reason, confidence = THEME_TO_SME[theme], "theme_map", "high"
        else:
            assignee, reason, confidence = FALLBACK_SME, "fallback", "low"

        question_set.append(
            {
                "question_id": f"Q-{i:03d}",
                "theme": theme,
                "question_text": gap["question"],
                "assignee_username": assignee,
                "assignment_reason": reason,
                "assignment_confidence": confidence,
                "status": "correction_required" if confidence == "low" else "pending",
            }
        )
    return question_set


# Standalone test
sample_gaps = [
    {"theme": "data_requirements", "question": "What is the source of the reset token expiry value?"},
    {"theme": "security_compliance", "question": "Should failed reset attempts be rate-limited?"},
    {"theme": "unmapped_theme", "question": "Who owns the email delivery SLA?"},
]
for q in generate_question_set(sample_gaps):
    print(q)


{'question_id': 'Q-001', 'theme': 'data_requirements', 'question_text': 'What is the source of the reset token expiry value?', 'assignee_username': 'priya.n', 'assignment_reason': 'theme_map', 'assignment_confidence': 'high', 'status': 'pending'}
{'question_id': 'Q-002', 'theme': 'security_compliance', 'question_text': 'Should failed reset attempts be rate-limited?', 'assignee_username': 'security-triage', 'assignment_reason': 'theme_map', 'assignment_confidence': 'high', 'status': 'pending'}
{'question_id': 'Q-003', 'theme': 'unmapped_theme', 'question_text': 'Who owns the email delivery SLA?', 'assignee_username': 'gov-triage', 'assignment_reason': 'fallback', 'assignment_confidence': 'low', 'status': 'correction_required'}


## 11. Authoring Tool 2/3 - `create_fsd_from_template`

Renders a normalized set of fields into the canonical FSD structure (Objective, Scope,
Actors, Requirements, Data Connectors, Snowflake Connectors, NFRs, Constraints, Open
Questions), as agreed for the FSD template.

In [6]:
FSD_SECTIONS = [
    "objective",
    "scope",
    "actors",
    "functional_requirements",
    "business_rules",
    "data_requirements",
    "data_connectors",
    "snowflake_connectors",
    "workflows",
    "non_functional_requirements",
    "acceptance_criteria",
    "constraints",
    "open_questions",
]


@tool
def create_fsd_from_template(sections: dict[str, Any]) -> str:
    """Render normalized FSD content into the canonical FSD template. `sections` maps
    section names (objective, scope, actors, functional_requirements, business_rules,
    data_requirements, data_connectors, snowflake_connectors, workflows,
    non_functional_requirements, acceptance_criteria, constraints, open_questions) to
    their content (string or list). Missing sections are rendered as empty rather than
    omitted, so the FSD structure is always consistent."""
    lines = ["# Functional Specification Document", ""]
    for section in FSD_SECTIONS:
        title = section.replace("_", " ").title()
        value = sections.get(section)
        lines.append(f"## {title}")
        if not value:
            lines.append("_None captured yet._")
        elif isinstance(value, list):
            for item in value:
                lines.append(f"- {item}")
        else:
            lines.append(str(value))
        lines.append("")
    return "\n".join(lines)


# Standalone test
sample_fsd = create_fsd_from_template(
    {
        "objective": "Allow users to securely reset a forgotten password via email.",
        "actors": ["End user", "Auth service", "Email delivery service"],
        "functional_requirements": [
            "REQ-1: User can request a password reset via email. [source: sample_requirements.docx]",
            "REQ-2: Reset link expires after 30 minutes. [source: sample_requirements.xlsx]",
        ],
        "data_connectors": ["User profile store (email address)"],
        "open_questions": ["Q-003: Who owns the email delivery SLA? (gov-triage, low confidence)"],
    }
)
print(sample_fsd)


# Functional Specification Document

## Objective
Allow users to securely reset a forgotten password via email.

## Scope
_None captured yet._

## Actors
- End user
- Auth service
- Email delivery service

## Functional Requirements
- REQ-1: User can request a password reset via email. [source: sample_requirements.docx]
- REQ-2: Reset link expires after 30 minutes. [source: sample_requirements.xlsx]

## Business Rules
_None captured yet._

## Data Requirements
_None captured yet._

## Data Connectors
- User profile store (email address)

## Snowflake Connectors
_None captured yet._

## Workflows
_None captured yet._

## Non Functional Requirements
_None captured yet._

## Acceptance Criteria
_None captured yet._

## Constraints
_None captured yet._

## Open Questions
- Q-003: Who owns the email delivery SLA? (gov-triage, low confidence)



## 12. Authoring Tool 3/3 - `score_completeness`

Deterministically checks that every requirement has linked evidence (a citation) and
that every open question is actually captured, rather than being silently skipped -
extends `src/tools/distill.py`'s existing `score_completeness` with the citation/open-
questions check discussed for this project.

In [7]:
@tool
def score_completeness(
    requirement_ids: list[str],
    evidence_requirement_ids: list[str],
    unresolved_question_ids: list[str],
    captured_open_question_ids: list[str],
) -> dict:
    """Deterministically score an FSD draft: the fraction of requirements that have a
    linked citation, and whether every unresolved question actually appears in the
    Open Questions section (not silently dropped). Call this with the ids you drafted
    before finalizing the FSD - do not estimate the score yourself."""
    missing_evidence = sorted(set(requirement_ids) - set(evidence_requirement_ids))
    missing_open_questions = sorted(set(unresolved_question_ids) - set(captured_open_question_ids))

    gaps = [f"requirement {r} has no linked citation" for r in missing_evidence]
    gaps += [f"question {q} is unresolved but missing from Open Questions" for q in missing_open_questions]

    total = len(requirement_ids) + len(unresolved_question_ids)
    if total == 0:
        return {"score": 1.0, "gaps": gaps}
    covered = (len(requirement_ids) - len(missing_evidence)) + (
        len(unresolved_question_ids) - len(missing_open_questions)
    )
    return {"score": round(covered / total, 2), "gaps": gaps}


# Standalone test
result = score_completeness(
    requirement_ids=["REQ-1", "REQ-2"],
    evidence_requirement_ids=["REQ-1"],
    unresolved_question_ids=["Q-003"],
    captured_open_question_ids=["Q-003"],
)
print(result)


{'score': 0.67, 'gaps': ['requirement REQ-2 has no linked citation']}


## 13. Assemble Tools into a Strands Agent

Wires all 10 tools into a real `strands.Agent` backed by `BedrockModel`, mirroring
`src/providers/agents/strands_factory.py` and the Distill system prompt in
`src/config/agents/distill.yaml`, updated for requirements-based input and the new
Data Connectors / Snowflake Connectors / Open Questions sections.

In [17]:
class Requirement(BaseModel):
    id: str
    description: str
    citation: str


class DistillFSD(BaseModel):
    """FSD output contract produced by the notebook's Distill agent."""

    title: str
    objective: str
    actors: list[str]
    functional_requirements: list[Requirement]
    data_connectors: list[str] = []
    snowflake_connectors: list[str] = []
    open_questions: list[str] = []
    completeness_score: float


DISTILL_SYSTEM_PROMPT = textwrap.dedent(
    """\
    You are the Distill agent in the Hyperdelivery factory. You take a raw requirements
    submission - which may include slide decks, Word docs, spreadsheets, PDFs, images,
    Jira tickets, or GitHub READMEs - and turn it into a Functional Specification
    Document (FSD).

    Use the ingestion tools (pptx_extract_text, docx_extract_text, xlsx_extract_data,
    pdf_extract_text, image_extract_text, jira_ticket_ingest, github_readme_ingest) when
    the prompt references a file path, issue key, or repo instead of raw text.

    If there are gaps or ambiguities, call generate_question_set with the gaps you found
    before finalizing the FSD - do not guess. Then call create_fsd_from_template to
    render the final FSD, including Data Connectors and Snowflake Connectors sections
    when the input touches those systems, and an Open Questions section for anything
    unresolved.

    Before finalizing, call score_completeness with the requirement ids, the requirement
    ids that have a citation, the unresolved question ids, and the question ids captured
    in Open Questions - use its returned score verbatim as completeness_score. Every
    requirement must carry a citation naming its source document/location.
    """
)

distill_tools = [
    pptx_extract_text,
    docx_extract_text,
    xlsx_extract_data,
    pdf_extract_text,
    image_extract_text,
    jira_ticket_ingest,
    github_readme_ingest,
    generate_question_set,
    create_fsd_from_template,
    score_completeness,
]

model = BedrockModel(
    model_id="global.anthropic.claude-sonnet-5",
    region_name=os.environ.get("AWS_REGION", "us-east-1"),
    max_tokens=8192,
)

distill_agent = Agent(model=model, system_prompt=DISTILL_SYSTEM_PROMPT, tools=distill_tools)
print(f"Distill agent assembled with {len(distill_tools)} tools.")


Distill agent assembled with 10 tools.


## 14. Run the Distill Agent End-to-End

Invokes the assembled agent against Amazon Bedrock with a prompt referencing the
generated sample files, and asks for a validated `DistillFSD` structured output -
requires working AWS credentials (e.g., `AWS_PROFILE=ua-hyperdelivery` already
configured for this repo).

In [19]:
prompt = f"""
The following requirement source files are available on disk:
- PowerPoint slide deck: {pptx_path}
- Word document: {docx_path}
- Excel sheet: {xlsx_path}
- PDF document: {pdf_path}
- Whiteboard photo (image): {image_path}

Read all of them using the appropriate ingestion tools, then produce a single,
de-duplicated Functional Specification Document for a "password reset via email"
feature. Every requirement must cite which source file it came from.
"""

result = distill_agent(prompt, structured_output_model=DistillFSD)

if result.structured_output is None:
    print("Agent did not produce structured output. Raw result:")
    print(result)
else:
    fsd = result.structured_output
    print(json.dumps(fsd.model_dump(), indent=2))



Tool #1: pptx_extract_text

Tool #2: docx_extract_text

Tool #3: xlsx_extract_data

Tool #4: pdf_extract_text

Tool #5: image_extract_text

Tool #6: generate_question_set
Now let's render the FSD and score completeness before finalizing.
Tool #7: create_fsd_from_template
Now let's score completeness.
Tool #8: score_completeness
All requirements are cited and all open questions are captured. Finalizing the FSD.
Tool #9: DistillFSD
{
  "title": "Password Reset via Email - Functional Specification Document",
  "objective": "Enable users to securely reset their forgotten password via a time-limited, single-use email link, ensuring delivery speed, auditability, and protection against account-enumeration abuse.",
  "actors": [
    "End User (account holder requesting reset)",
    "Password Reset Service (backend system)",
    "Email Delivery Service (outbound email provider)",
    "Auditor/Compliance Reviewer (consumes audit logs)"
  ],
  "functional_requirements": [
    {
      "id": "FR-1